# 08_Temporal_Intelligence_Feature_Engineering

Generate Momentum, Volatility, Growth Consistency, CAGR and Trend Slope features.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression

master_df = pd.read_csv('../Generated Datasets/master_skill_dataset_v7.csv')
master_df.head()

## Historical Demand Dataset\nExpected columns: skill, year, demand

In [ ]:
historical_df = pd.read_csv('../Generated Datasets/skill_demand_history.csv')
historical_df.head()

In [ ]:
rows=[]

for skill, group in historical_df.groupby('skill'):
    group=group.sort_values('year')
    years=group['year'].values.reshape(-1,1)
    demand=group['demand'].values

    if len(demand)<2:
        continue

    growth_rates=np.diff(demand)/np.maximum(demand[:-1],1)

    momentum_score=growth_rates[-1] if len(growth_rates)>0 else 0
    volatility_score=np.std(growth_rates) if len(growth_rates)>1 else 0
    growth_consistency=np.sum(growth_rates>0)/len(growth_rates) if len(growth_rates)>0 else 0

    start_val=max(demand[0],1)
    end_val=max(demand[-1],1)
    years_diff=max(group['year'].max()-group['year'].min(),1)

    cagr=((end_val/start_val)**(1/years_diff))-1

    lr=LinearRegression()
    lr.fit(years,demand)
    trend_slope=lr.coef_[0]

    rows.append([skill,momentum_score,volatility_score,growth_consistency,cagr,trend_slope])

features=pd.DataFrame(rows,columns=['skill','momentum_score','volatility_score','growth_consistency','cagr','trend_slope'])
features.head()

In [ ]:
for col in ['momentum_score','volatility_score','growth_consistency','cagr','trend_slope']:
    mn=features[col].min()
    mx=features[col].max()
    if mx!=mn:
        features[col]=(features[col]-mn)/(mx-mn)

In [ ]:
master_v8=master_df.merge(features,on='skill',how='left')
master_v8.head()

In [ ]:
master_v8.to_csv('../Generated Datasets/master_skill_dataset_v8.csv',index=False)
print(master_v8.shape)
print('Saved master_skill_dataset_v8.csv')